# 02 · Correlation & regression

How features co-vary across geographies. Rank correlation (Spearman) is the default because most
urban count features are heavily right-skewed. Coefficients here are descriptive and cross-sectional,
not causal; watch for confounding by area, population and land use.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from city_fabric import analysis as cfa

sns.set_theme(style="whitegrid", context="notebook", font_scale=0.9)
pd.set_option("display.max_columns", 50, "display.width", 200, "display.float_format", "{:,.4g}".format)

In [ ]:
LEVEL = "nta"
df = cfa.load_features(LEVEL)
cat = cfa.catalog()
# Density and rate features are comparable across geographies of different size; raw counts are not.
# Rates are comparable across geographies of different size and population; raw counts are not.
RATE_UNITS = ("fraction", "days", "inches", "species", "USD", "minutes", "points", "violations", "cuisines")
FEATURES = [f for f in cat.index if cat.loc[f, "kind"] in ("per_km2", "per_capita", "share")
            or (cat.loc[f, "kind"] == "raw" and cat.loc[f, "unit"] in RATE_UNITS)]
len(FEATURES)

## Correlation matrix (clustered)

In [ ]:
core = [f for f in FEATURES if not f.startswith(("sr311_type__", "restaurants_cuisine__"))
        and cat.loc[f, "kind"] != "per_km2"] + ["acs_population_per_km2"]
corr = cfa.corr_matrix(df, core, method="spearman").dropna(how="all").dropna(axis=1, how="all")
labels = cat.loc[corr.index, "title"].str.slice(0, 45)
g = sns.clustermap(corr.fillna(0), cmap="RdBu_r", vmin=-1, vmax=1, figsize=(12, 12),
                   xticklabels=labels, yticklabels=labels, dendrogram_ratio=0.08, cbar_pos=(0.02, 0.85, 0.02, 0.1))
g.ax_heatmap.tick_params(labelsize=7)

## Strongest pairs
Pairs that share a base feature (e.g. a count and its density) are excluded.

In [ ]:
pairs = cfa.top_pairs(cfa.corr_matrix(df, FEATURES), n=25)
pairs.assign(a_title=cat.loc[pairs["a"], "title"].values, b_title=cat.loc[pairs["b"], "title"].values)[["a_title", "b_title", "r"]]

## Density confound
Per-km² features inherit population density: denser places have more of everything. Compare each
density feature's correlation with population density against its per-capita counterpart.

In [ ]:
rows = []
for f in cfa.feature_ids(kind="per_capita"):
    base = cat.loc[f, "base_feature"]
    dens = f"{base}_per_km2"
    if dens in df.columns and not base.startswith(("sr311_type__",)):
        rows.append({"feature": cat.loc[base, "title"],
                     "rho(per km², pop density)": df[dens].corr(df["acs_population_per_km2"], method="spearman"),
                     "rho(per 1k, pop density)": df[f].corr(df["acs_population_per_km2"], method="spearman")})
pd.DataFrame(rows).set_index("feature").round(2)

## Focus: tree canopy vs. income

In [ ]:
X, Y = "acs_median_hh_income", "trees_live_per_1k"
d = df[[X, Y, "borough"]].dropna()
d = d[(d[X] > 0) & (d[Y] > 0)]
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
sns.scatterplot(data=d, x=X, y=Y, hue="borough", s=18, ax=axes[0])
axes[0].set(xscale="log", yscale="log", xlabel=cat.loc[X, "title"], ylabel=cat.loc[Y, "title"])
sns.regplot(x=np.log10(d[X]), y=np.log10(d[Y]), ax=axes[1], scatter_kws={"s": 10, "alpha": .5}, line_kws={"color": "C3"})
axes[1].set(xlabel=f"log10 {X}", ylabel=f"log10 {Y}")
fig.tight_layout()

## Regression

Log-log OLS with HC3 robust standard errors and borough fixed effects: 311 requests per resident on
tree supply, controlling for income, population density and renter cost burden. Log coefficients are
elasticities (% change in requests per resident per 1% change in the regressor).

In [ ]:
import statsmodels.formula.api as smf

d = df.copy()
for c in ["sr311_total_per_1k", "trees_live_per_1k", "trees_planted_5y_per_1k", "acs_median_hh_income", "acs_population_per_km2"]:
    d[f"log_{c}"] = np.log(d[c].where(d[c] > 0))
MODEL = ("log_sr311_total_per_1k ~ log_trees_live_per_1k + log_trees_planted_5y_per_1k + log_acs_median_hh_income"
         " + log_acs_population_per_km2 + acs_pct_rent_burdened + C(borough)")
d = d.dropna(subset=[c for c in d.columns if c.startswith("log_")] + ["acs_pct_rent_burdened", "borough"])
fit = smf.ols(MODEL, data=d).fit(cov_type="HC3")
fit.summary()

## Standardized coefficients
All variables z-scored, so coefficients are comparable effect sizes (in SDs).

In [ ]:
Y = "sr311_median_days_to_close"
Xs = ["acs_median_hh_income", "acs_population_per_km2", "acs_pct_commute_transit", "sr311_total_per_1k", "sr311_pct_open"]
res = cfa.regress(df, Y, Xs, log=["acs_median_hh_income", "acs_population_per_km2", "sr311_total_per_1k"], standardize=True)
coef = pd.DataFrame({"beta": res.params, "se": res.bse, "p": res.pvalues}).drop("const")
ax = coef["beta"].plot.barh(xerr=1.96 * coef["se"], color="#3b6fb6", figsize=(6, 3))
ax.axvline(0, color="k", lw=.8); ax.set_title(f"Standardized effects on {Y}  (R²={res.rsquared:.2f}, n={int(res.nobs)})", fontsize=9, loc="left")
coef

## Residual map
Where does the model under/over-predict? Spatially clustered residuals indicate missing covariates.

In [ ]:
resid = pd.Series(fit.resid, index=d.index)
ax = cfa.choropleth(LEVEL, resid, cmap="RdBu_r", scheme="quantile", title="Residuals: log 311 requests per resident")

## Robustness across levels
Does income vs. trees per resident hold when aggregated differently? (Modifiable areal unit problem.)

In [ ]:
logpos = lambda s: np.log(s.where(s > 0))
rows = []
for lv in cfa.levels():
    t = cfa.load_features(lv)
    if len(t) < 10:
        continue
    rows.append({"level": lv, "n": len(t),
                 "spearman": t["acs_median_hh_income"].corr(t["trees_live_per_1k"], method="spearman"),
                 "pearson_log": logpos(t["acs_median_hh_income"]).corr(logpos(t["trees_live_per_1k"]))})
pd.DataFrame(rows)